In [1]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 1 — SETUP, FEATURE ENGINEERING, ENCODING      ║
# ║  V2: Stronger features + GroupKFold by Race         ║
# ╚══════════════════════════════════════════════════════╝

import numpy as np
import pandas as pd
import warnings, time, os, json
warnings.filterwarnings("ignore")

from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import OrdinalEncoder

SAVE_DIR      = "C:/Users/hp/OneDrive/Desktop/predicting pitstop/Pit--Stop-prediction/pipeline_cache"

TUNE_SAMPLE   = 0.20
FINAL_FOLDS   = 5
EARLY_STOP    = 50
LGB_MAX_TREES = 3000
XGB_MAX_TREES = 3000
CAT_MAX_TREES = 3000

os.makedirs(SAVE_DIR, exist_ok=True)

train = pd.read_csv("C:/Users/hp/OneDrive/Desktop/predicting pitstop/Pit--Stop-prediction/dataset/train.csv")
test  = pd.read_csv("C:/Users/hp/OneDrive/Desktop/predicting pitstop/Pit--Stop-prediction/dataset/test.csv")
print(f"Train: {train.shape}  |  Test: {test.shape}")
print(f"Columns: {train.columns.tolist()}")

# ── FEATURE ENGINEERING ─────────────────────────────────
def engineer_features(df):
    df = df.copy().sort_values(['Race','Driver','LapNumber']).reset_index(drop=True)

    id_col = df['id'].copy() if 'id' in df.columns else None

    grp     = df.groupby(['Race','Driver'])
    grp_race = df.groupby('Race')

    # ── 1. LAP TIME FEATURES ────────────────────────────
    df['LapTime_Lag1'] = grp['LapTime (s)'].shift(1)
    df['LapTime_Lag2'] = grp['LapTime (s)'].shift(2)
    df['LapTime_Lag3'] = grp['LapTime (s)'].shift(3)
    df['LapTime_Lag4'] = grp['LapTime (s)'].shift(4)
    df['LapTime_Lag5'] = grp['LapTime (s)'].shift(5)

    for c in ['LapTime_Lag1','LapTime_Lag2','LapTime_Lag3','LapTime_Lag4','LapTime_Lag5']:
        df[c] = df[c].fillna(df['LapTime (s)'])

    df['LapTime_Roll3_Mean'] = grp['LapTime (s)'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    df['LapTime_Roll5_Mean'] = grp['LapTime (s)'].transform(lambda x: x.rolling(5, min_periods=1).mean())
    df['LapTime_Roll10_Mean']= grp['LapTime (s)'].transform(lambda x: x.rolling(10,min_periods=1).mean())
    df['LapTime_Roll3_Std']  = grp['LapTime (s)'].transform(lambda x: x.rolling(3, min_periods=1).std().fillna(0))
    df['LapTime_Roll5_Std']  = grp['LapTime (s)'].transform(lambda x: x.rolling(5, min_periods=1).std().fillna(0))
    df['LapTime_Roll3_Max']  = grp['LapTime (s)'].transform(lambda x: x.rolling(3, min_periods=1).max())
    df['LapTime_Roll3_Min']  = grp['LapTime (s)'].transform(lambda x: x.rolling(3, min_periods=1).min())

    # Clip extreme lap times per race
    q1  = grp_race['LapTime (s)'].transform(lambda x: x.quantile(0.25))
    q3  = grp_race['LapTime (s)'].transform(lambda x: x.quantile(0.75))
    df['LapTime (s)'] = df['LapTime (s)'].clip(upper=q3 + 1.5*(q3-q1))

    df['LapTime_Trend']         = df['LapTime (s)'] - df['LapTime_Roll3_Mean']
    df['LapTime_Trend5']        = df['LapTime (s)'] - df['LapTime_Roll5_Mean']
    df['LapTime_Acceleration']  = df['LapTime_Trend'] - grp['LapTime_Trend'].shift(1).fillna(0)
    df['LapTime_vs_PB']         = df['LapTime (s)'] - grp['LapTime (s)'].transform('min')

    # Consecutive slow laps (pace degradation streak)
    is_slow = (df['LapTime_Trend'] > 0).astype(int)
    df['Consecutive_Slow_Laps'] = grp['LapTime_Trend'].transform(
        lambda x: x.gt(0).groupby((x.gt(0) != x.gt(0).shift()).cumsum()).cumcount() + 1
    ).where(is_slow == 1, 0)

    # ── 2. POSITION FEATURES ────────────────────────────
    df['Position_Lag1']         = grp['Position'].shift(1).fillna(df['Position'])
    df['Position_Lag2']         = grp['Position'].shift(2).fillna(df['Position'])
    df['Position_Change_1Lap']  = df['Position'] - df['Position_Lag1']
    df['Position_Change_2Laps'] = df['Position'] - df['Position_Lag2']
    df['Position_Change_3Laps'] = grp['Position'].diff(3).fillna(0)
    df['Position_Change_5Laps'] = grp['Position'].diff(5).fillna(0)
    df['Positions_Gained']      = df['Position_Change_1Lap'].clip(upper=0).abs()
    df['Positions_Lost']        = df['Position_Change_1Lap'].clip(lower=0)
    df['Is_Points_Position']    = (df['Position'] <= 10).astype(int)
    df['Is_Podium_Position']    = (df['Position'] <= 3).astype(int)
    df['Is_Top5_Position']      = (df['Position'] <= 5).astype(int)

    # ── 3. TYRE / DEGRADATION FEATURES ──────────────────
    df['LapTime_Delta_abs']     = df['LapTime_Delta'].abs()
    df['Degradation_Pace']      = df['Cumulative_Degradation'] * df['LapTime_Delta_abs']
    df['Avg_Wear_Rate']         = df['Cumulative_Degradation'] / (df['TyreLife'] + 1e-5)
    df['Wear_Squared']          = df['Cumulative_Degradation'] ** 2
    df['TyreLife_Squared']      = df['TyreLife'] ** 2
    df['Wear_x_TyreLife']       = df['Cumulative_Degradation'] * df['TyreLife']
    df['Cumulative_Deg_Roll3']  = grp['Cumulative_Degradation'].transform(
                                      lambda x: x.rolling(3,min_periods=1).mean())
    df['Cumulative_Deg_Roll5']  = grp['Cumulative_Degradation'].transform(
                                      lambda x: x.rolling(5,min_periods=1).mean())
    df['Deg_Acceleration']      = grp['Cumulative_Degradation'].diff(1).fillna(0)
    df['Deg_Acceleration2']     = grp['Deg_Acceleration'].diff(1).fillna(0)  # 2nd derivative
    df['Deg_Lag1']              = grp['Cumulative_Degradation'].shift(1).fillna(0)
    df['Deg_Lag3']              = grp['Cumulative_Degradation'].shift(3).fillna(0)

    # ── 4. STINT / PIT WINDOW FEATURES ──────────────────
    # Expected stint length per compound (domain knowledge)
    compound_stint = {'SOFT': 20, 'MEDIUM': 30, 'HARD': 40,
                      'INTERMEDIATE': 25, 'WET': 20}
    if 'Compound' in df.columns:
        df['Expected_Stint_Length'] = df['Compound'].map(compound_stint).fillna(25)
        df['Tyre_Life_Ratio']       = df['TyreLife'] / df['Expected_Stint_Length']
        df['Laps_Past_Expected']    = (df['TyreLife'] - df['Expected_Stint_Length']).clip(lower=0)
        df['In_Pit_Window']         = (
            (df['TyreLife'] >= df['Expected_Stint_Length'] * 0.7) &
            (df['TyreLife'] <= df['Expected_Stint_Length'] * 1.3)
        ).astype(int)
    else:
        df['Expected_Stint_Length'] = 25
        df['Tyre_Life_Ratio']       = df['TyreLife'] / 25
        df['Laps_Past_Expected']    = (df['TyreLife'] - 25).clip(lower=0)
        df['In_Pit_Window']         = ((df['TyreLife'] >= 17) & (df['TyreLife'] <= 32)).astype(int)

    # ── 5. RACE PROGRESS FEATURES ───────────────────────
    df['Max_Laps']        = grp['LapNumber'].transform('max')
    df['Race_Progress']   = df['LapNumber'] / df['Max_Laps']
    df['Laps_Remaining']  = df['Max_Laps'] - df['LapNumber']
    df['Late_Race_Flag']  = (df['Race_Progress'] > 0.80).astype(int)
    df['Early_Race_Flag'] = (df['Race_Progress'] < 0.20).astype(int)
    df['Mid_Race_Flag']   = ((df['Race_Progress'] >= 0.35) & (df['Race_Progress'] <= 0.65)).astype(int)
    # Pit becomes impossible in last 5 laps
    df['Too_Late_To_Pit'] = (df['Laps_Remaining'] <= 5).astype(int)

    # ── 6. PACE GAP FEATURES ────────────────────────────
    # Gap to race leader on same lap
    best_lap_in_race = df.groupby(['Race','LapNumber'])['LapTime (s)'].transform('min')
    df['Gap_To_Leader_Lap']  = df['LapTime (s)'] - best_lap_in_race
    # Gap to own best lap this race
    df['Gap_To_Own_Best']    = df['LapTime (s)'] - grp['LapTime (s)'].transform('min')
    # Gap to median lap time in race (normalised pace)
    median_lap = grp_race['LapTime (s)'].transform('median')
    df['Normalised_Pace']    = df['LapTime (s)'] / (median_lap + 1e-5)
    # Rolling pace vs field
    field_roll3 = df.groupby(['Race','LapNumber'])['LapTime (s)'].transform('mean')
    df['Pace_vs_Field']      = df['LapTime (s)'] - field_roll3

    # ── 7. INTERACTION FEATURES ─────────────────────────
    df['Wear_x_PositionLoss']     = df['Cumulative_Degradation'] * df['Positions_Lost']
    df['Age_x_Trend']             = df['TyreLife'] * df['LapTime_Trend']
    df['Progress_x_Wear']         = df['Race_Progress'] * df['Cumulative_Degradation']
    df['Points_x_Wear']           = df['Is_Points_Position'] * df['Cumulative_Degradation']
    df['TyreLifeRatio_x_Trend']   = df['Tyre_Life_Ratio'] * df['LapTime_Trend']
    df['PitWindow_x_Deg']         = df['In_Pit_Window'] * df['Cumulative_Degradation']
    df['LapsRemaining_x_Wear']    = df['Laps_Remaining'] * df['Cumulative_Degradation']
    df['PastExpected_x_Trend']    = df['Laps_Past_Expected'] * df['LapTime_Trend']

    # ── 8. HISTORICAL PIT STATS PER DRIVER/RACE ─────────
    # Average stint length this driver has done so far this race
    df['Driver_Avg_Stint_So_Far'] = grp['TyreLife'].transform(
        lambda x: x.expanding().mean()
    )
    # How many laps since the driver's last stint started
    # (TyreLife resets at pit so it IS the laps on current tyre)
    df['Laps_On_Tyre_Norm'] = df['TyreLife'] / (df['Max_Laps'] / 2 + 1e-5)

    cols_to_drop = ['id','PitStop','Driver','RaceProgress',
                    'Pace_Loss','Is_Points_Position','Positions_Lost']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    return df, id_col

print("Engineering features ...")
t0 = time.time()
train_eng, _        = engineer_features(train)
test_eng,  test_ids = engineer_features(test)
print(f"  Done in {time.time()-t0:.1f}s")
print(f"  Features: {train_eng.shape[1]}")

assert len(test_ids) == len(test_eng), "ID/feature row count mismatch!"
print(f"  test_ids aligned: {len(test_ids):,} rows ✓")

# ── TARGET ENCODING ─────────────────────────────────────
def target_encode(train_df, test_df, columns, target, n_splits=5):
    train_df    = train_df.copy()
    test_df     = test_df.copy()
    global_mean = train_df[target].mean()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=2026)
    for col in columns:
        train_df[f'{col}_TE'] = global_mean
        for tr_idx, val_idx in kf.split(train_df):
            means = train_df.iloc[tr_idx].groupby(col)[target].mean()
            train_df.loc[train_df.iloc[val_idx].index, f'{col}_TE'] = \
                train_df.loc[train_df.iloc[val_idx].index, col].map(means)
        means_full = train_df.groupby(col)[target].mean()
        test_df[f'{col}_TE'] = test_df[col].map(means_full).fillna(global_mean)
    train_df = train_df.drop(columns=columns)
    test_df  = test_df.drop(columns=columns)
    return train_df, test_df

# Target encode Race AND Compound
encode_cols = [c for c in ['Race','Compound','TeamName'] if c in train_eng.columns]
print(f"Target encoding: {encode_cols}")
train_eng, test_eng = target_encode(train_eng, test_eng, encode_cols, 'PitNextLap')
print(f"After encoding shape: {train_eng.shape}")

# ── DROP HIGHLY CORRELATED ──────────────────────────────
corr  = train_eng.drop(columns=['PitNextLap']).corr(numeric_only=True).abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop_corr = [c for c in upper.columns if any(upper[c] > 0.97)]
print(f"Dropping {len(to_drop_corr)} correlated features: {to_drop_corr}")
train_eng = train_eng.drop(columns=to_drop_corr)
test_eng  = test_eng.drop(columns=[c for c in to_drop_corr if c in test_eng.columns])

# ── PREPARE X, y ────────────────────────────────────────
TARGET = 'PitNextLap'
X      = train_eng.drop(columns=[TARGET]).reset_index(drop=True)
y      = train_eng[TARGET].astype(int).reset_index(drop=True)

# Keep Race column for GroupKFold — will be dropped before model training
race_groups = X['Race_TE'].copy() if 'Race_TE' in X.columns else None

X_test = test_eng.reindex(columns=X.columns, fill_value=0).reset_index(drop=True)

X      = X.fillna(X.median(numeric_only=True))
X_test = X_test.fillna(X.median(numeric_only=True))

# ── ENCODE REMAINING OBJECT COLUMNS ─────────────────────
obj_cols = X.select_dtypes(include='object').columns.tolist()
print(f"Object columns to encode: {obj_cols}")
if obj_cols:
    compound_order = ['SOFT','MEDIUM','HARD','INTERMEDIATE','WET']
    categories = []
    for col in obj_cols:
        categories.append(compound_order if col == 'Compound'
                          else sorted(X[col].dropna().unique().tolist()))
    oe = OrdinalEncoder(categories=categories,
                        handle_unknown='use_encoded_value', unknown_value=-1)
    X[obj_cols]      = oe.fit_transform(X[obj_cols])
    X_test[obj_cols] = oe.transform(X_test[obj_cols])

X      = X.astype(np.float32)
X_test = X_test.astype(np.float32)

SCALE_POS = float((y==0).sum()) / float((y==1).sum())
print(f"\nFinal shape      : {X.shape}")
print(f"Positive rate    : {y.mean():.4f}")
print(f"scale_pos_weight : {SCALE_POS:.2f}")

# ── ALIGNMENT CHECK ─────────────────────────────────────
assert len(X_test) == len(test_ids), \
    f"MISMATCH: X_test={len(X_test)}, test_ids={len(test_ids)}"
print(f"Alignment check  : ✓")

# ── TUNE SAMPLE INDICES ─────────────────────────────────
tune_idx, _ = train_test_split(
    np.arange(len(X)), train_size=TUNE_SAMPLE, stratify=y, random_state=2026
)

# ── SAVE ────────────────────────────────────────────────
print("\nSaving ...")
np.save(f"{SAVE_DIR}/X.npy",        X.values)
np.save(f"{SAVE_DIR}/X_test.npy",   X_test.values)
np.save(f"{SAVE_DIR}/y.npy",        y.values)
np.save(f"{SAVE_DIR}/tune_idx.npy", tune_idx)

col_names = X.columns.tolist()
with open(f"{SAVE_DIR}/col_names.json", "w") as f:
    json.dump(col_names, f)

test_ids.reset_index(drop=True).to_frame(name='id').to_csv(
    f"{SAVE_DIR}/test_ids.csv", index=False
)

config = dict(
    USE_GPU=False, TUNE_SAMPLE=TUNE_SAMPLE, FINAL_FOLDS=FINAL_FOLDS,
    EARLY_STOP=EARLY_STOP, LGB_MAX_TREES=LGB_MAX_TREES,
    XGB_MAX_TREES=XGB_MAX_TREES, CAT_MAX_TREES=CAT_MAX_TREES,
    SAVE_DIR=SAVE_DIR, SCALE_POS=SCALE_POS,
)
with open(f"{SAVE_DIR}/config.json", "w") as f:
    json.dump(config, f)

print(f"  X.npy      : {X.shape}  ({X.memory_usage(deep=True).sum()/1e6:.0f} MB)")
print(f"  X_test.npy : {X_test.shape}")
print(f"  Features   : {len(col_names)}")
print(f"\n✓ Saved to {SAVE_DIR}")
print("  → Run CELL 2 (LightGBM tuning)")

Train: (439140, 16)  |  Test: (188165, 15)
Columns: ['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'PitNextLap']
Engineering features ...
  Done in 40.9s
  Features: 75
  test_ids aligned: 188,165 rows ✓
Target encoding: ['Race', 'Compound']
After encoding shape: (439140, 75)
Dropping 2 correlated features: ['Gap_To_Own_Best', 'TyreLifeRatio_x_Trend']
Object columns to encode: []

Final shape      : (439140, 72)
Positive rate    : 0.1990
scale_pos_weight : 4.03
Alignment check  : ✓

Saving ...
  X.npy      : (439140, 72)  (126 MB)
  X_test.npy : (188165, 72)
  Features   : 72

✓ Saved to C:/Users/hp/OneDrive/Desktop/predicting pitstop/Pit--Stop-prediction/pipeline_cache
  → Run CELL 2 (LightGBM tuning)


In [2]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 2 — TUNE LIGHTGBM                             ║
# ║  Prerequisite: Cell 1 must be done.                  ║
# ║  Tuning on CPU (avoids VRAM overflow).               ║
# ╚══════════════════════════════════════════════════════╝

import numpy as np
import pandas as pd
import json, pickle, time
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

with open("C:/Users/hp/OneDrive/Desktop/predicting pitstop/Pit--Stop-prediction/pipeline_cache/config.json") as f:
    cfg = json.load(f)

SAVE_DIR      = cfg['SAVE_DIR']
USE_GPU       = cfg['USE_GPU']
SCALE_POS     = cfg['SCALE_POS']
EARLY_STOP    = cfg['EARLY_STOP']
MAX_TREES     = cfg['LGB_MAX_TREES']
OPTUNA_TRIALS = 30

with open(f"{SAVE_DIR}/col_names.json") as f:
    col_names = json.load(f)

X   = pd.DataFrame(np.load(f"{SAVE_DIR}/X.npy"), columns=col_names)
y   = pd.Series(np.load(f"{SAVE_DIR}/y.npy"), name='PitNextLap')
tune_idx = np.load(f"{SAVE_DIR}/tune_idx.npy")

X_tune = X.iloc[tune_idx].reset_index(drop=True)
y_tune = y.iloc[tune_idx].reset_index(drop=True)
print(f"Tune sample : {len(X_tune):,} rows  (CPU)")
print(f"Full data   : {len(X):,} rows  (GPU for final step)")

# CPU for Optuna trials, GPU only for final n_estimators calibration
LGB_TUNE  = {}
LGB_FINAL = dict(device='gpu', gpu_platform_id=0, gpu_device_id=0) if USE_GPU else {}

def objective_lgb(trial):
    params = dict(
        max_depth         = trial.suggest_int  ('max_depth',          3,   10),
        num_leaves        = trial.suggest_int  ('num_leaves',        15,  200),
        learning_rate     = trial.suggest_float('learning_rate',   0.01,  0.3, log=True),
        min_child_samples = trial.suggest_int  ('min_child_samples', 10,  100),
        subsample         = trial.suggest_float('subsample',         0.5,  1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree',  0.5,  1.0),
        reg_alpha         = trial.suggest_float('reg_alpha',        1e-4, 10.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda',       1e-4, 10.0, log=True),
    )
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_tune, y_tune, test_size=0.2, stratify=y_tune, random_state=2026
    )
    model = LGBMClassifier(
        **params, **LGB_TUNE,
        n_estimators=MAX_TREES, scale_pos_weight=SCALE_POS,
        random_state=2026, n_jobs=-1, verbose=-1,
    )
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[early_stopping(EARLY_STOP, verbose=False), log_evaluation(-1)])
    return roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])

print(f"\nTuning LightGBM ({OPTUNA_TRIALS} trials on CPU) ...")
t0 = time.time()
study = optuna.create_study(direction='maximize')
study.optimize(objective_lgb, n_trials=OPTUNA_TRIALS,
               show_progress_bar=True, catch=(Exception,))
print(f"  Best AUC    : {study.best_value:.5f}  ({time.time()-t0:.0f}s)")
print(f"  Best params : {study.best_params}")
print(f"  Completed   : {sum(t.state.name=='COMPLETE' for t in study.trials)}/{len(study.trials)} trials")

print(f"\nFinding best n_estimators on full {len(X):,} rows ({'GPU' if USE_GPU else 'CPU'}) ...")
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
final_model = LGBMClassifier(
    **study.best_params, **LGB_FINAL,
    n_estimators=MAX_TREES, scale_pos_weight=SCALE_POS,
    random_state=2026, n_jobs=-1, verbose=-1,
)
final_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                callbacks=[early_stopping(EARLY_STOP, verbose=False), log_evaluation(100)])
best_n = final_model.best_iteration_
print(f"  Best n_estimators : {best_n}")

result = {'best_params': study.best_params, 'best_n_estimators': best_n, 'tune_auc': study.best_value}
with open(f"{SAVE_DIR}/lgb_tuning_result.pkl", 'wb') as f:
    pickle.dump(result, f)

print(f"\n✓ Saved → {SAVE_DIR}/lgb_tuning_result.pkl")
print("  → Rest, then run CELL 3 (XGBoost tuning)")

Tune sample : 87,828 rows  (CPU)
Full data   : 439,140 rows  (GPU for final step)

Tuning LightGBM (30 trials on CPU) ...


Best trial: 24. Best value: 0.942092: 100%|██████████| 30/30 [04:38<00:00,  9.27s/it]


  Best AUC    : 0.94209  (278s)
  Best params : {'max_depth': 8, 'num_leaves': 121, 'learning_rate': 0.010019014312833188, 'min_child_samples': 80, 'subsample': 0.6947691799734892, 'colsample_bytree': 0.5458668554247237, 'reg_alpha': 0.8222614577507846, 'reg_lambda': 1.581819377098704}
  Completed   : 30/30 trials

Finding best n_estimators on full 439,140 rows (CPU) ...
[100]	valid_0's binary_logloss: 0.342607
[200]	valid_0's binary_logloss: 0.3275
[300]	valid_0's binary_logloss: 0.325801
[400]	valid_0's binary_logloss: 0.321615
[500]	valid_0's binary_logloss: 0.317016
[600]	valid_0's binary_logloss: 0.313366
[700]	valid_0's binary_logloss: 0.310327
[800]	valid_0's binary_logloss: 0.308027
[900]	valid_0's binary_logloss: 0.306296
[1000]	valid_0's binary_logloss: 0.304753
[1100]	valid_0's binary_logloss: 0.303341
[1200]	valid_0's binary_logloss: 0.302044
[1300]	valid_0's binary_logloss: 0.300782
[1400]	valid_0's binary_logloss: 0.299564
[1500]	valid_0's binary_logloss: 0.298466
[1600]	

In [3]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 3 — TUNE XGBOOST                              ║
# ║  Prerequisite: Cell 1 must be done.                  ║
# ╚══════════════════════════════════════════════════════╝

import numpy as np
import pandas as pd
import json, pickle, time
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

with open("C:/Users/hp/OneDrive/Desktop/predicting pitstop/Pit--Stop-prediction/pipeline_cache/config.json") as f:
    cfg = json.load(f)

SAVE_DIR      = cfg['SAVE_DIR']
USE_GPU       = cfg['USE_GPU']
SCALE_POS     = cfg['SCALE_POS']
EARLY_STOP    = cfg['EARLY_STOP']
MAX_TREES     = cfg['XGB_MAX_TREES']
OPTUNA_TRIALS = 30

with open(f"{SAVE_DIR}/col_names.json") as f:
    col_names = json.load(f)

X   = pd.DataFrame(np.load(f"{SAVE_DIR}/X.npy"), columns=col_names)
y   = pd.Series(np.load(f"{SAVE_DIR}/y.npy"), name='PitNextLap')
tune_idx = np.load(f"{SAVE_DIR}/tune_idx.npy")

X_tune = X.iloc[tune_idx].reset_index(drop=True)
y_tune = y.iloc[tune_idx].reset_index(drop=True)
print(f"Tune sample : {len(X_tune):,} rows  |  Full: {len(X):,} rows")

XGB_GPU = dict(device='cuda') if USE_GPU else dict(tree_method='hist')

def objective_xgb(trial):
    params = dict(
        max_depth        = trial.suggest_int  ('max_depth',          3,   10),
        learning_rate    = trial.suggest_float('learning_rate',   0.01,  0.3, log=True),
        min_child_weight = trial.suggest_int  ('min_child_weight',   1,   20),
        subsample        = trial.suggest_float('subsample',          0.5,  1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree',   0.5,  1.0),
        gamma            = trial.suggest_float('gamma',              0.0,  5.0),
        reg_alpha        = trial.suggest_float('reg_alpha',         1e-4, 10.0, log=True),
        reg_lambda       = trial.suggest_float('reg_lambda',        1e-4, 10.0, log=True),
    )
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_tune, y_tune, test_size=0.2, stratify=y_tune, random_state=2026
    )
    model = XGBClassifier(
        **params, **XGB_GPU,
        n_estimators=MAX_TREES, early_stopping_rounds=EARLY_STOP,
        scale_pos_weight=SCALE_POS, eval_metric='logloss',
        use_label_encoder=False, random_state=2026, n_jobs=-1,
    )
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    return roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])

print(f"\nTuning XGBoost ({OPTUNA_TRIALS} trials) ...")
t0 = time.time()
study = optuna.create_study(direction='maximize')
study.optimize(objective_xgb, n_trials=OPTUNA_TRIALS,
               show_progress_bar=True, catch=(Exception,))
print(f"  Best AUC    : {study.best_value:.5f}  ({time.time()-t0:.0f}s)")
print(f"  Best params : {study.best_params}")

print(f"\nFinding best n_estimators on full {len(X):,} rows ...")
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
final_model = XGBClassifier(
    **study.best_params, **XGB_GPU,
    n_estimators=MAX_TREES, early_stopping_rounds=EARLY_STOP,
    scale_pos_weight=SCALE_POS, eval_metric='logloss',
    use_label_encoder=False, random_state=2026, n_jobs=-1,
)
final_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=100)
best_n = final_model.best_iteration
print(f"  Best n_estimators : {best_n}")

result = {'best_params': study.best_params, 'best_n_estimators': best_n, 'tune_auc': study.best_value}
with open(f"{SAVE_DIR}/xgb_tuning_result.pkl", 'wb') as f:
    pickle.dump(result, f)

print(f"\n✓ Saved → {SAVE_DIR}/xgb_tuning_result.pkl")
print("  → Rest, then run CELL 4 (CatBoost tuning)")

Tune sample : 87,828 rows  |  Full: 439,140 rows

Tuning XGBoost (30 trials) ...


Best trial: 24. Best value: 0.942411: 100%|██████████| 30/30 [10:47<00:00, 21.59s/it]


  Best AUC    : 0.94241  (648s)
  Best params : {'max_depth': 8, 'learning_rate': 0.025269970630494982, 'min_child_weight': 12, 'subsample': 0.8237644909949137, 'colsample_bytree': 0.5050548939524675, 'gamma': 3.17279381773525, 'reg_alpha': 9.27681116876355, 'reg_lambda': 0.0012080949450679489}

Finding best n_estimators on full 439,140 rows ...
[0]	validation_0-logloss:0.68382
[100]	validation_0-logloss:0.35456
[200]	validation_0-logloss:0.32043
[300]	validation_0-logloss:0.30975
[400]	validation_0-logloss:0.30442
[500]	validation_0-logloss:0.30073
[600]	validation_0-logloss:0.29754
[700]	validation_0-logloss:0.29516
[800]	validation_0-logloss:0.29323
[900]	validation_0-logloss:0.29133
[1000]	validation_0-logloss:0.28961
[1100]	validation_0-logloss:0.28817
[1200]	validation_0-logloss:0.28686
[1300]	validation_0-logloss:0.28563
[1400]	validation_0-logloss:0.28470
[1500]	validation_0-logloss:0.28374
[1600]	validation_0-logloss:0.28298
[1700]	validation_0-logloss:0.28217
[1800]	validatio

In [4]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 4 — TUNE CATBOOST                             ║
# ║  Prerequisite: Cell 1 must be done.                  ║
# ╚══════════════════════════════════════════════════════╝

import numpy as np
import pandas as pd
import json, pickle, time
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

with open("C:/Users/hp/OneDrive/Desktop/predicting pitstop/Pit--Stop-prediction/pipeline_cache/config.json") as f:
    cfg = json.load(f)

SAVE_DIR      = cfg['SAVE_DIR']
USE_GPU       = cfg['USE_GPU']
EARLY_STOP    = cfg['EARLY_STOP']
MAX_TREES     = cfg['CAT_MAX_TREES']
OPTUNA_TRIALS = 20

with open(f"{SAVE_DIR}/col_names.json") as f:
    col_names = json.load(f)

X   = pd.DataFrame(np.load(f"{SAVE_DIR}/X.npy"), columns=col_names)
y   = pd.Series(np.load(f"{SAVE_DIR}/y.npy"), name='PitNextLap')
tune_idx = np.load(f"{SAVE_DIR}/tune_idx.npy")

X_tune = X.iloc[tune_idx].reset_index(drop=True)
y_tune = y.iloc[tune_idx].reset_index(drop=True)
print(f"Tune sample : {len(X_tune):,} rows  |  Full: {len(X):,} rows")

CAT_GPU     = dict(task_type='GPU', devices='0') if USE_GPU else {}
CAT_BALANCE = dict(auto_class_weights='Balanced')

def objective_cat(trial):
    params = dict(
        depth               = trial.suggest_int  ('depth',                4,   10),
        learning_rate       = trial.suggest_float('learning_rate',     0.01,  0.3, log=True),
        l2_leaf_reg         = trial.suggest_float('l2_leaf_reg',        1.0, 10.0),
        bagging_temperature = trial.suggest_float('bagging_temperature', 0.0,  1.0),
    )
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_tune, y_tune, test_size=0.2, stratify=y_tune, random_state=2026
    )
    model = CatBoostClassifier(
        **params, **CAT_GPU, **CAT_BALANCE,
        iterations=MAX_TREES, early_stopping_rounds=EARLY_STOP,
        eval_metric='AUC', random_seed=2026, verbose=False,
    )
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False)
    return roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])

print(f"\nTuning CatBoost ({OPTUNA_TRIALS} trials) ...")
t0 = time.time()
study = optuna.create_study(direction='maximize')
study.optimize(objective_cat, n_trials=OPTUNA_TRIALS,
               show_progress_bar=True, catch=(Exception,))
print(f"  Best AUC    : {study.best_value:.5f}  ({time.time()-t0:.0f}s)")
print(f"  Best params : {study.best_params}")

print(f"\nFinding best iterations on full {len(X):,} rows ...")
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
final_model = CatBoostClassifier(
    **study.best_params, **CAT_GPU, **CAT_BALANCE,
    iterations=MAX_TREES, early_stopping_rounds=EARLY_STOP,
    eval_metric='AUC', random_seed=2026, verbose=100,
)
final_model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=100)
best_n = final_model.best_iteration_
print(f"  Best iterations : {best_n}")

result = {'best_params': study.best_params, 'best_n_estimators': best_n, 'tune_auc': study.best_value}
with open(f"{SAVE_DIR}/cat_tuning_result.pkl", 'wb') as f:
    pickle.dump(result, f)

print(f"\n✓ Saved → {SAVE_DIR}/cat_tuning_result.pkl")
print("  → Rest, then run CELL 5 (OOF LightGBM)")

Tune sample : 87,828 rows  |  Full: 439,140 rows

Tuning CatBoost (20 trials) ...


Best trial: 13. Best value: 0.94116: 100%|██████████| 20/20 [17:58<00:00, 53.90s/it]


  Best AUC    : 0.94116  (1078s)
  Best params : {'depth': 8, 'learning_rate': 0.018145687983053535, 'l2_leaf_reg': 7.079812776090304, 'bagging_temperature': 0.5502618535039647}

Finding best iterations on full 439,140 rows ...
0:	test: 0.9018243	best: 0.9018243 (0)	total: 76.5ms	remaining: 3m 49s
100:	test: 0.9216524	best: 0.9216524 (100)	total: 7.99s	remaining: 3m 49s
200:	test: 0.9272009	best: 0.9272009 (200)	total: 15.9s	remaining: 3m 41s
300:	test: 0.9305312	best: 0.9305312 (300)	total: 23.4s	remaining: 3m 29s
400:	test: 0.9327792	best: 0.9327792 (400)	total: 31s	remaining: 3m 20s
500:	test: 0.9345886	best: 0.9345886 (500)	total: 38.6s	remaining: 3m 12s
600:	test: 0.9360691	best: 0.9360691 (600)	total: 46.3s	remaining: 3m 4s
700:	test: 0.9371932	best: 0.9371932 (700)	total: 53.9s	remaining: 2m 56s
800:	test: 0.9383541	best: 0.9383541 (800)	total: 1m 1s	remaining: 2m 49s
900:	test: 0.9394154	best: 0.9394154 (900)	total: 1m 9s	remaining: 2m 41s
1000:	test: 0.9400580	best: 0.9400580 

In [5]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 5 — OOF TRAINING: LIGHTGBM                   ║
# ║  Prerequisite: Cells 1 + 2 must be done.            ║
# ╚══════════════════════════════════════════════════════╝

import numpy as np
import pandas as pd
import json, pickle, time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

with open("C:/Users/hp/OneDrive/Desktop/predicting pitstop/Pit--Stop-prediction/pipeline_cache/config.json") as f:
    cfg = json.load(f)

SAVE_DIR    = cfg['SAVE_DIR']
USE_GPU     = cfg['USE_GPU']
SCALE_POS   = cfg['SCALE_POS']
FINAL_FOLDS = cfg['FINAL_FOLDS']

with open(f"{SAVE_DIR}/col_names.json") as f:
    col_names = json.load(f)

X      = pd.DataFrame(np.load(f"{SAVE_DIR}/X.npy"),      columns=col_names)
X_test = pd.DataFrame(np.load(f"{SAVE_DIR}/X_test.npy"), columns=col_names)
y      = pd.Series(np.load(f"{SAVE_DIR}/y.npy"), name='PitNextLap')

with open(f"{SAVE_DIR}/lgb_tuning_result.pkl", 'rb') as f:
    lgb_result = pickle.load(f)

best_params = lgb_result['best_params']
best_n      = lgb_result['best_n_estimators']
LGB_GPU     = dict(device='gpu', gpu_platform_id=0, gpu_device_id=0) if USE_GPU else {}

print(f"LGB params       : {best_params}")
print(f"LGB n_estimators : {best_n}")

SKF             = StratifiedKFold(n_splits=FINAL_FOLDS, shuffle=True, random_state=2026)
oof             = np.zeros(len(X))
test_fold_preds = np.zeros((len(X_test), FINAL_FOLDS))

t0 = time.time()
for fold, (tr_idx, val_idx) in enumerate(SKF.split(X, y)):
    print(f"  Fold {fold+1}/{FINAL_FOLDS} ...", end=' ')
    model = LGBMClassifier(
        **best_params, **LGB_GPU,
        n_estimators=best_n, scale_pos_weight=SCALE_POS,
        random_state=2026, n_jobs=-1, verbose=-1,
    )
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    oof[val_idx]             = model.predict_proba(X.iloc[val_idx])[:, 1]
    test_fold_preds[:, fold] = model.predict_proba(X_test)[:, 1]
    print(f"AUC: {roc_auc_score(y.iloc[val_idx], oof[val_idx]):.5f}")

print(f"\n  LGB OOF AUC : {roc_auc_score(y, oof):.5f}  ({time.time()-t0:.0f}s)")

np.save(f"{SAVE_DIR}/oof_lgb.npy",  oof)
np.save(f"{SAVE_DIR}/test_lgb.npy", test_fold_preds.mean(axis=1))
print(f"\n✓ Saved → oof_lgb.npy  |  → Rest, then run CELL 6 (OOF XGBoost)")

LGB params       : {'max_depth': 8, 'num_leaves': 121, 'learning_rate': 0.010019014312833188, 'min_child_samples': 80, 'subsample': 0.6947691799734892, 'colsample_bytree': 0.5458668554247237, 'reg_alpha': 0.8222614577507846, 'reg_lambda': 1.581819377098704}
LGB n_estimators : 3000
  Fold 1/5 ... AUC: 0.94657
  Fold 2/5 ... AUC: 0.94546
  Fold 3/5 ... AUC: 0.94637
  Fold 4/5 ... AUC: 0.94616
  Fold 5/5 ... AUC: 0.94609

  LGB OOF AUC : 0.94612  (381s)

✓ Saved → oof_lgb.npy  |  → Rest, then run CELL 6 (OOF XGBoost)


In [6]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 6 — OOF TRAINING: XGBOOST                    ║
# ║  Prerequisite: Cells 1 + 3 must be done.            ║
# ╚══════════════════════════════════════════════════════╝

import numpy as np
import pandas as pd
import json, pickle, time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

with open("C:/Users/hp/OneDrive/Desktop/predicting pitstop/Pit--Stop-prediction/pipeline_cache/config.json") as f:
    cfg = json.load(f)

SAVE_DIR    = cfg['SAVE_DIR']
USE_GPU     = cfg['USE_GPU']
SCALE_POS   = cfg['SCALE_POS']
FINAL_FOLDS = cfg['FINAL_FOLDS']

with open(f"{SAVE_DIR}/col_names.json") as f:
    col_names = json.load(f)

X      = pd.DataFrame(np.load(f"{SAVE_DIR}/X.npy"),      columns=col_names)
X_test = pd.DataFrame(np.load(f"{SAVE_DIR}/X_test.npy"), columns=col_names)
y      = pd.Series(np.load(f"{SAVE_DIR}/y.npy"), name='PitNextLap')

with open(f"{SAVE_DIR}/xgb_tuning_result.pkl", 'rb') as f:
    xgb_result = pickle.load(f)

best_params = xgb_result['best_params']
best_n      = xgb_result['best_n_estimators']
XGB_GPU     = dict(device='cuda') if USE_GPU else dict(tree_method='hist')

print(f"XGB params       : {best_params}")
print(f"XGB n_estimators : {best_n}")

SKF             = StratifiedKFold(n_splits=FINAL_FOLDS, shuffle=True, random_state=2026)
oof             = np.zeros(len(X))
test_fold_preds = np.zeros((len(X_test), FINAL_FOLDS))

t0 = time.time()
for fold, (tr_idx, val_idx) in enumerate(SKF.split(X, y)):
    print(f"  Fold {fold+1}/{FINAL_FOLDS} ...", end=' ')
    model = XGBClassifier(
        **best_params, **XGB_GPU,
        n_estimators=best_n, scale_pos_weight=SCALE_POS,
        eval_metric='logloss', use_label_encoder=False,
        random_state=2026, n_jobs=-1,
    )
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    oof[val_idx]             = model.predict_proba(X.iloc[val_idx])[:, 1]
    test_fold_preds[:, fold] = model.predict_proba(X_test)[:, 1]
    print(f"AUC: {roc_auc_score(y.iloc[val_idx], oof[val_idx]):.5f}")

print(f"\n  XGB OOF AUC : {roc_auc_score(y, oof):.5f}  ({time.time()-t0:.0f}s)")

np.save(f"{SAVE_DIR}/oof_xgb.npy",  oof)
np.save(f"{SAVE_DIR}/test_xgb.npy", test_fold_preds.mean(axis=1))
print(f"\n✓ Saved → oof_xgb.npy  |  → Rest, then run CELL 7 (OOF CatBoost)")

XGB params       : {'max_depth': 8, 'learning_rate': 0.025269970630494982, 'min_child_weight': 12, 'subsample': 0.8237644909949137, 'colsample_bytree': 0.5050548939524675, 'gamma': 3.17279381773525, 'reg_alpha': 9.27681116876355, 'reg_lambda': 0.0012080949450679489}
XGB n_estimators : 2991
  Fold 1/5 ... AUC: 0.94625
  Fold 2/5 ... AUC: 0.94545
  Fold 3/5 ... AUC: 0.94631
  Fold 4/5 ... AUC: 0.94609
  Fold 5/5 ... AUC: 0.94623

  XGB OOF AUC : 0.94606  (425s)

✓ Saved → oof_xgb.npy  |  → Rest, then run CELL 7 (OOF CatBoost)


In [7]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 7 — OOF TRAINING: CATBOOST                   ║
# ║  Prerequisite: Cells 1 + 4 must be done.            ║
# ╚══════════════════════════════════════════════════════╝

import numpy as np
import pandas as pd
import json, pickle, time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

with open("C:/Users/hp/OneDrive/Desktop/predicting pitstop/Pit--Stop-prediction/pipeline_cache/config.json") as f:
    cfg = json.load(f)

SAVE_DIR    = cfg['SAVE_DIR']
USE_GPU     = cfg['USE_GPU']
FINAL_FOLDS = cfg['FINAL_FOLDS']

with open(f"{SAVE_DIR}/col_names.json") as f:
    col_names = json.load(f)

X      = pd.DataFrame(np.load(f"{SAVE_DIR}/X.npy"),      columns=col_names)
X_test = pd.DataFrame(np.load(f"{SAVE_DIR}/X_test.npy"), columns=col_names)
y      = pd.Series(np.load(f"{SAVE_DIR}/y.npy"), name='PitNextLap')

with open(f"{SAVE_DIR}/cat_tuning_result.pkl", 'rb') as f:
    cat_result = pickle.load(f)

best_params = cat_result['best_params']
best_n      = cat_result['best_n_estimators']
CAT_GPU     = dict(task_type='GPU', devices='0') if USE_GPU else {}
CAT_BALANCE = dict(auto_class_weights='Balanced')

print(f"CAT params     : {best_params}")
print(f"CAT iterations : {best_n}")

SKF             = StratifiedKFold(n_splits=FINAL_FOLDS, shuffle=True, random_state=2026)
oof             = np.zeros(len(X))
test_fold_preds = np.zeros((len(X_test), FINAL_FOLDS))

t0 = time.time()
for fold, (tr_idx, val_idx) in enumerate(SKF.split(X, y)):
    print(f"  Fold {fold+1}/{FINAL_FOLDS} ...", end=' ')
    model = CatBoostClassifier(
        **best_params, **CAT_GPU, **CAT_BALANCE,
        iterations=best_n, random_seed=2026, verbose=False,
    )
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    oof[val_idx]             = model.predict_proba(X.iloc[val_idx])[:, 1]
    test_fold_preds[:, fold] = model.predict_proba(X_test)[:, 1]
    print(f"AUC: {roc_auc_score(y.iloc[val_idx], oof[val_idx]):.5f}")

print(f"\n  CAT OOF AUC : {roc_auc_score(y, oof):.5f}  ({time.time()-t0:.0f}s)")

np.save(f"{SAVE_DIR}/oof_cat.npy",  oof)
np.save(f"{SAVE_DIR}/test_cat.npy", test_fold_preds.mean(axis=1))
print(f"\n✓ Saved → oof_cat.npy  |  → Rest, then run CELL 8 (Stack + Submit)")

CAT params     : {'depth': 8, 'learning_rate': 0.018145687983053535, 'l2_leaf_reg': 7.079812776090304, 'bagging_temperature': 0.5502618535039647}
CAT iterations : 2998
  Fold 1/5 ... AUC: 0.94551
  Fold 2/5 ... AUC: 0.94440
  Fold 3/5 ... AUC: 0.94510
  Fold 4/5 ... AUC: 0.94502
  Fold 5/5 ... AUC: 0.94526

  CAT OOF AUC : 0.94505  (873s)

✓ Saved → oof_cat.npy  |  → Rest, then run CELL 8 (Stack + Submit)


In [ ]:
# from scipy.optimize import minimize
# from sklearn.metrics import roc_auc_score

# # Define the objective: Maximize AUC (so we minimize the negative AUC)
# def objective(weights):
#     weights = weights / np.sum(weights) # Ensure weights sum to 1
#     blend = (weights[0] * oof_xgb) + (weights[1] * oof_lgb) + (weights[2] * oof_cat)
#     return -roc_auc_score(y, blend)

# # Initial guess (equal weighting)
# starting_weights = [0.33, 0.33, 0.33]

# # Run the optimizer
# res = minimize(objective, starting_weights, method='Nelder-Mead')
# best_w = res.x / np.sum(res.x)

# print(f"Optimal Weights -> XGB: {best_w[0]:.3f}, LGB: {best_w[1]:.3f}, CAT: {best_w[2]:.3f}")

# # Check the new Best OOF AUC
# best_blend = (best_w[0] * oof_xgb) + (best_w[1] * oof_lgb) + (best_w[2] * oof_cat)
# print(f"Optimized Blend OOF AUC: {roc_auc_score(y, best_blend):.5f}")

In [8]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 8 — STACK + SUBMISSION                        ║
# ║  Prerequisite: ALL previous cells must be done.     ║
# ║  This cell is very fast (< 1 min).                   ║
# ╚══════════════════════════════════════════════════════╝

import numpy as np
import pandas as pd
import json
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score

with open("C:/Users/hp/OneDrive/Desktop/predicting pitstop/Pit--Stop-prediction/pipeline_cache/config.json") as f:
    cfg = json.load(f)

SAVE_DIR    = cfg['SAVE_DIR']
FINAL_FOLDS = cfg['FINAL_FOLDS']

y = pd.Series(np.load(f"{SAVE_DIR}/y.npy"), name='PitNextLap')

# ── Load test_ids in the SAME sorted order as X_test ───
# (saved by Cell 1 after sorting by Race/Driver/LapNumber)
test_ids = pd.read_csv(f"{SAVE_DIR}/test_ids.csv")
print(f"test_ids shape : {test_ids.shape}")
print(f"Sample ids     : {test_ids['id'].values[:5]}")

# ── LOAD ALL OOF + TEST PREDS ───────────────────────────
oof_lgb  = np.load(f"{SAVE_DIR}/oof_lgb.npy")
oof_xgb  = np.load(f"{SAVE_DIR}/oof_xgb.npy")
oof_cat  = np.load(f"{SAVE_DIR}/oof_cat.npy")
test_lgb = np.load(f"{SAVE_DIR}/test_lgb.npy")
test_xgb = np.load(f"{SAVE_DIR}/test_xgb.npy")
test_cat = np.load(f"{SAVE_DIR}/test_cat.npy")

# ── ALIGNMENT CHECK ─────────────────────────────────────
assert len(test_lgb) == len(test_ids), \
    f"MISMATCH: predictions={len(test_lgb)} rows, test_ids={len(test_ids)} rows!"
print(f"Alignment check : predictions ({len(test_lgb):,}) == test_ids ({len(test_ids):,}) ✓")

print("\nIndividual OOF AUCs:")
print(f"  LGB : {roc_auc_score(y, oof_lgb):.5f}")
print(f"  XGB : {roc_auc_score(y, oof_xgb):.5f}")
print(f"  CAT : {roc_auc_score(y, oof_cat):.5f}")

# ── STACK ───────────────────────────────────────────────
oof_stack  = np.column_stack([oof_lgb,  oof_xgb,  oof_cat])
test_stack = np.column_stack([test_lgb, test_xgb, test_cat])

scaler      = StandardScaler()
oof_scaled  = scaler.fit_transform(oof_stack)
test_scaled = scaler.transform(test_stack)

SKF        = StratifiedKFold(n_splits=FINAL_FOLDS, shuffle=True, random_state=2026)
meta_model = LogisticRegression(C=0.1, class_weight='balanced',
                                 solver='lbfgs', max_iter=1000, random_state=2026)
meta_oof   = cross_val_predict(meta_model, oof_scaled, y,
                                cv=SKF, method='predict_proba')[:, 1]

print(f"\n{'='*50}")
print(f"STACKED META-MODEL OOF AUC : {roc_auc_score(y, meta_oof):.5f}")
print(f"{'='*50}")

# ── THRESHOLD OPTIMISATION ──────────────────────────────
thresholds = np.arange(0.05, 0.95, 0.01)
f1s        = [f1_score(y, (meta_oof >= t).astype(int)) for t in thresholds]
best_thr   = thresholds[int(np.argmax(f1s))]
print(f"Best threshold : {best_thr:.2f}  |  Best OOF F1 : {max(f1s):.5f}")

# ── FINAL PREDICTIONS ───────────────────────────────────
meta_model.fit(oof_scaled, y)
final_proba  = meta_model.predict_proba(test_scaled)[:, 1]
final_binary = (final_proba >= best_thr).astype(int)

# ── BUILD SUBMISSION ────────────────────────────────────
# test_ids is already in the same row order as X_test / predictions
# so we just zip them together directly — no merge/sort needed
submission = pd.DataFrame({
    "id"        : test_ids['id'].values,   # sorted order from Cell 1
    "PitNextLap": final_proba,             # same sorted order from OOF cells
    # "PitNextLap": final_binary,          # uncomment if metric is F1 not AUC
})

# ── SANITY CHECK before saving ──────────────────────────
assert submission['id'].nunique() == len(submission), \
    "Duplicate IDs found in submission — something went wrong!"
assert submission['PitNextLap'].between(0, 1).all(), \
    "Probabilities outside [0,1] — check predictions!"
print(f"\nSanity checks passed ✓")
print(f"  Unique IDs      : {submission['id'].nunique():,}")
print(f"  Prob range      : [{submission['PitNextLap'].min():.4f}, {submission['PitNextLap'].max():.4f}]")
print(f"  Predicted pit % : {(final_binary==1).mean()*100:.2f}%")

submission.to_csv("C:/Users/hp/OneDrive/Desktop/predicting pitstop/Pit--Stop-prediction/submission.csv", index=False)
print(f"\n✓ Submission saved!  Shape: {submission.shape}")
print(submission.head(10))

test_ids shape : (188165, 1)
Sample ids     : [463410 496081 452309 476804 449143]
Alignment check : predictions (188,165) == test_ids (188,165) ✓

Individual OOF AUCs:
  LGB : 0.94612
  XGB : 0.94606
  CAT : 0.94505

STACKED META-MODEL OOF AUC : 0.94631
Best threshold : 0.76  |  Best OOF F1 : 0.74844

Sanity checks passed ✓
  Unique IDs      : 188,165
  Prob range      : [0.0406, 0.9589]
  Predicted pit % : 23.36%

✓ Submission saved!  Shape: (188165, 2)
       id  PitNextLap
0  463410    0.133001
1  496081    0.050915
2  452309    0.226817
3  476804    0.049966
4  449143    0.685090
5  573714    0.157555
6  546731    0.706515
7  475620    0.044820
8  616284    0.042426
9  450232    0.095063
